# 강의 06 · 실습 12 — 패턴 6 평가자-최적화자 · (3) 변형

## 1. 문제상황

- 병원 원무과는 진료 예약이 잡히면 환자에게 안내 문자를 보냅니다.
- 문자는 70자를 넘으면 두 통으로 쪼개져 나가므로 70자 이내여야 하고, 예약 날짜·시간·진료과·장소가 빠지면 안 됩니다.
- 담당자는 문자 초안을 에이전트 하나에 맡겼는데, 자세히 쓰면 글자 수를 넘기고 짧게 쓰면 장소가 빠집니다.
- 글자 수를 세고 빠진 항목을 확인해 다시 시키는 일이 예약마다 반복됩니다.

## 2. 문제와 목표

- **문제**: 문자 초안을 만드는 일과, 글자 수 규칙과 필수 내용 두 가지 기준으로 판정하는 일을 한 호출이 같이 하므로, 두 기준을 함께 지킨 초안이 나올 때까지 사람이 확인해야 합니다.
- **목표**: 예약 정보를 입력하면 generator 노드가 문자 초안을 만들고, evaluator 노드가 글자 수는 코드 규칙으로, 필수 내용은 모델로 판정해 반려 사유를 실어 되돌리고, 합격이거나 시도 상한에 닿으면 끝나는 처리 흐름을 만듭니다. 시도마다 판정 이력을 상태에 남깁니다.
    - generator: 예약 안내 문자 초안 — 첫 시도는 자세히 세 문장, 지적이 있으면 70자 이내 한 문장으로 다시 씁니다.
    - evaluator 두 단: 코드가 글자 수(70자)를 먼저 채점하고, 통과한 초안만 모델이 날짜·시간·진료과·장소 유무를 `Verdict`로 판정합니다.
    - 판정 이력: 상태의 `history` 리스트(시도마다 「N회차: 판정(단)」).
- **목표 달성 여부의 판정 기준**: 예약 정보를 입력했을 때, 첫 초안이 글자 수 규칙에서 반려되고, 두 번째 초안이 70자 이내이면서 필수 내용 넷을 담아 합격하는 것을 실행 결과에서 확인합니다. 최종 상태의 판정 이력에 두 항목이 순서대로 남습니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex12_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 평가 규격을 정의합니다.**
    - 예약 정보(`booking`), 문자 초안(`sms`), 지적(`feedback`), 판정(`grade`), 시도 횟수(`tries`), 판정 이력(`history`) 키 여섯 개를 가지는 상태를 선언합니다.
    - `history`에는 `operator.add` 리듀서를 걸어 시도마다 항목이 덧붙게 합니다.
    - 평가 규격 `Verdict`는 `grade` 값을 합격·반려 둘로 제한하고, `feedback` 필드에 반려라면 고칠 점 한 문장, 합격이면 빈 문자열을 담습니다.
    - 글자 수 상한은 70, 시도 상한은 3으로 둡니다.
2. **generator 노드를 만듭니다.**
    - 지적이 비어 있으면 「예약 안내 문자를 자세히 세 문장으로 쓴다」는 지침으로 예약 정보만 넣어 첫 초안을 만듭니다.
    - 지적이 있으면 「지적을 반영해 예약 안내 문자를 70자 이내 한 문장으로 다시 쓴다」는 지침으로 지적과 예약 정보를 함께 넣어 다시 씁니다.
    - 결과를 `sms`에 쓰고 `tries`를 1 올립니다.
3. **evaluator 노드를 만듭니다.**
    - 먼저 코드로 `sms`의 글자 수를 셉니다.
    - 70자를 넘으면 모델을 부르지 않고 반려로 판정하며, 지적에는 실제 글자 수와 「날짜·시간·진료과·장소만 남기고 70자 이내로 줄인다」를 적습니다.
    - 70자 이내면 `Verdict` 규격을 건 모델에 「문자에 예약 날짜·시간·진료과·장소가 모두 있으면 합격, 아니면 반려」라는 지침으로 예약 정보와 초안을 넣어 판정을 받습니다.
    - 어느 단에서 판정했든 `grade`·`feedback`에 쓰고, 「시도 번호: 판정(단)」 항목(단은 「규칙 채점」 또는 「모델 채점」)을 `history`에 덧붙입니다.
4. **그래프에 노드를 등록합니다.**
    - generator·evaluator 두 노드를 이름과 함께 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 generator로, generator에서 evaluator로 가는 고정 엣지를 추가합니다.
    - evaluator 뒤에는 판단 함수 route_sms가 합격이거나 `tries`가 상한이면 END를, 아니면 generator를 돌려주는 조건부 엣지를 추가합니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 예약 정보와 빈 초안, 빈 지적, 빈 판정, 시도 횟수 0, 빈 이력을 넣어 실행한 뒤, 최종 판정과 시도 횟수와 판정 이력과 문자를 출력합니다.
    - 값(`BOOKING`, 예약 정보)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - evaluator는 규칙 채점에서 반려하면 「[evaluator] 진입 -> 1단 규칙 채점 반려 (N자)」 줄을, 모델 채점을 하면 「[evaluator] 진입 -> 2단 모델 채점 (합격 또는 반려)」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 평가 규격을 선언하고 이력 키에 리듀서를 겁니다 | `class SmsState(TypedDict)`, `Annotated[list, operator.add]`, `class Verdict(BaseModel)` | 1 |
| ② 노드 함수 정의 | 초안을 만드는 generator와 규칙·모델 두 단으로 판정하는 evaluator를 만듭니다 | `def generator(state) -> dict`, `len(sms) > MAX_CHARS`, `llm.with_structured_output(Verdict)` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 두 노드를 등록합니다 | `StateGraph(SmsState)`, `add_node` | 4 |
| ④ 엣지 연결 | 직렬 순서와 되돌림 조건을 정합니다 | `add_edge`, `add_conditional_edges` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 예약 정보를 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import operator
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field   # Field — 규격 필드에 설명(description=…)을 달 때 씁니다

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 예약 정보 BOOKING — 값을 그대로 씁니다
BOOKING = ("10월 15일(수) 오후 2시 30분 정형외과 김민수 교수 진료. "
           "본관 3층 접수처에서 접수. 15분 전 도착, 신분증 지참.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 두 노드가 함께 읽고 쓰는 키를 선언합니다. 예약 정보·초안·지적·판정·시도 횟수 키 다섯 개에 판정 이력 `history`가 더해집니다. evaluator가 시도마다 항목 하나를 돌려주므로 `operator.add` 리듀서를 걸어 덮이지 않고 쌓이게 합니다. 글자 수 상한 `MAX_CHARS`는 코드가 세는 규칙이고, 시도 상한 `MAX_TRIES`는 루프를 멈추는 규칙입니다.

In [ ]:
# 여기에 단계 ①(평가 규격 Verdict와 상태 정의, 글자 수 상한·시도 상한 상수)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- generator는 첫 시도와 다시 쓰기를 함께 맡습니다. 첫 시도는 일부러 자세히 세 문장으로 시켜 글자 수 규칙에 걸리도록 합니다.
- evaluator는 두 단입니다. 코드가 셀 수 있는 것(글자 수)은 모델에게 묻지 않습니다. 규칙에 걸리면 바로 반려하고 지적에 실제 글자 수를 적습니다. 규칙을 통과한 초안만 모델이 필수 내용을 판정합니다.
- 어느 단에서 판정했든 `history`에 「시도 번호: 판정(단)」을 덧붙입니다. 어느 기준에서 반려됐는지가 이력에 남습니다.

In [ ]:
# 여기에 단계 ②(generator 노드와 두 단 evaluator 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 생성 노드와 평가 노드 둘뿐입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `route_sms`가 종료 조건 자체입니다. 합격이거나 시도 상한에 닿으면 END, 아니면 생성 노드로 되돌립니다. 상한이 없으면 반려가 이어질 때 루프가 스스로 멈추지 못합니다.

In [ ]:
# 여기에 단계 ④(되돌림 판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 예약 정보와 빈 값들을 넣으면 최종 상태가 돌아옵니다. 아래에서는 정형외과 진료 예약 정보를 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 1회차 초안은 70자를 넘고, `[evaluator]` 줄에 「1단 규칙 채점 반려」와 실제 글자 수가 출력됩니다.
2. 2회차 초안은 70자 이내이고, `[evaluator]` 줄에 「2단 모델 채점 합격」이 출력됩니다.
3. 최종 상태의 `history`는 「1회차: 반려(규칙 채점)」「2회차: 합격(모델 채점)」 두 항목입니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. `history`에 항목이 하나만 남으면 단계 ①의 리듀서를, 1회차가 모델 채점으로 넘어가면 단계 ②의 글자 수 규칙 순서를 다시 봅니다.